# Pipeline de Preprocesamiento — PaySim Fraud Detection

**Proyecto**: Sentinel-fraud-ml
**Autor**: Edwin Carlo
**Fecha**: Octubre 2026
**Notebook previo**: [`01_eda.ipynb`](./01_eda.ipynb)
**Módulo reutilizable**: [`src/preprocessing.py`](../src/preprocessing.py)

---

## Objetivo del notebook

Construir y aplicar el pipeline de preprocesamiento que transforma el dataset crudo de PaySim en conjuntos de features listos para el modelado supervisado (Paso 4).

## Estrategia

Este notebook **no implementa lógica directamente**. Toda la lógica está en funciones puras y testeables del módulo `src/preprocessing.py`. El notebook:

1. **Importa** las funciones desde `src/`.
2. **Orquesta** el pipeline paso a paso.
3. **Verifica** cada transformación con métricas y visualizaciones.
4. **Guarda** los conjuntos resultantes en `data/processed/`.

Esta separación de responsabilidades permite testing unitario, reutilización y reproducibilidad.

## Pipeline


In [1]:
# --- Librerías estándar ---
import sys
from pathlib import Path

# --- Análisis de datos ---
import numpy as np
import pandas as pd

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Módulo del proyecto ---
sys.path.append("..")  # para poder importar desde src/
from src.preprocessing import (
    load_data,
    filter_transactions,
    drop_unused_columns,
    engineer_features,
    split_features_target,
    split_train_val_test,
    VALID_TYPES,
    TARGET_COL,
    RANDOM_SEED,
)

# --- Configuración global ---
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# --- Rutas ---
DATA_RAW = Path("..") / "data" / "raw" / "PS_20174392719_1491204439457_log.csv"
DATA_PROCESSED = Path("..") / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"Tipos válidos: {VALID_TYPES}")
print(f"Variable objetivo: {TARGET_COL}")

Python: 3.12.7
Pandas: 2.2.2
Tipos válidos: ['TRANSFER', 'CASH_OUT']
Variable objetivo: isFraud


---

## 1. Carga del dataset crudo

**Objetivo**: Cargar el CSV de PaySim desde `data/raw/` usando la función `load_data()` del módulo.

**Por qué importa**: la carga se centraliza en una función que valida la existencia del archivo y que no esté vacío. Esto permite detectar problemas de inmediato.

In [2]:
df_raw = load_data(DATA_RAW)

print(f"✅ Dataset cargado")
print(f"   Filas:    {df_raw.shape[0]:>12,}")
print(f"   Columnas: {df_raw.shape[1]:>12,}")
print(f"   Memoria:  {df_raw.memory_usage(deep=True).sum() / 1024**2:>10.2f} MB")

✅ Dataset cargado
   Filas:       6,362,620
   Columnas:           11
   Memoria:     1452.57 MB


**Hallazgo**: el dataset se carga correctamente desde el CSV crudo.

**Decisión**: proceder al filtrado por tipo de transacción según los hallazgos del EDA.

**Nota**: en esta etapa aún no hay transformaciones aplicadas. El DataFrame `df_raw` se mantiene intacto para auditoría.

---

## 2. Filtrado por tipo de transacción

**Objetivo**: Reducir el dataset a solo `TRANSFER` y `CASH_OUT`, donde ocurre el 100% del fraude según el EDA.

**Justificación**:

- Los tipos `CASH-IN`, `DEBIT` y `PAYMENT` tienen tasa de fraude = 0%.
- Filtrarlos:
  - Reduce el tamaño del dataset (de ~6.36M a ~2.77M filas).
  - Elimina ruido que no aporta señal al modelo.
  - Acelera el entrenamiento.
- No se pierde ningún caso positivo (fraude).

In [3]:
df_filtered = filter_transactions(df_raw, valid_types=VALID_TYPES)

print(f"Filas antes:  {len(df_raw):>12,}")
print(f"Filas después: {len(df_filtered):>12,}")
print(f"Reducción:    {(1 - len(df_filtered) / len(df_raw)) * 100:>11.2f}%")

# Verificar que el fraude se preserva completo
fraudes_antes = df_raw[TARGET_COL].sum()
fraudes_despues = df_filtered[TARGET_COL].sum()
print(f"\nFraudes antes:  {fraudes_antes:>12,}")
print(f"Fraudes después: {fraudes_despues:>12,}")
print(f"¿Se preservan todos? {'✅ Sí' if fraudes_antes == fraudes_despues else '❌ NO'}")

Filas antes:     6,362,620
Filas después:    2,770,409
Reducción:          56.46%

Fraudes antes:         8,213
Fraudes después:        8,213
¿Se preservan todos? ✅ Sí


**Hallazgo**: el filtrado reduce significativamente el tamaño del dataset sin perder ningún fraude.

**Verificación crítica**: `fraudes_antes == fraudes_despues` confirma que el filtro no elimina casos positivos.

**Decisión**: mantener el dataset filtrado como base para los siguientes pasos.

---

## 3. Eliminación de columnas no informativas

**Objetivo**: Eliminar columnas que no aportan al modelo o que introducen sesgo.

**Columnas eliminadas**:

| Columna | Motivo |
|---------|--------|
| `nameOrig` | Identificador único del cliente; no generalizable. |
| `nameDest` | Identificador único del receptor; no generalizable (aunque de él se derivará `is_merchant_dest` antes de eliminarlo). |
| `isFlaggedFraud` | Regla de negocio con cobertura ~0.2%; redundante con la etiqueta. |

**Nota**: `nameDest` se elimina **después** de crear `is_merchant_dest` para preservar la información de si el receptor es comercio o cliente.

In [4]:
# IMPORTANTE: primero crear features que dependen de nameDest
df_feat = engineer_features(df_filtered)

# Verificar nuevas columnas
new_cols = [c for c in df_feat.columns if c not in df_filtered.columns]
print("Nuevas features creadas:")
for c in new_cols:
    print(f"  - {c}")

Nuevas features creadas:
  - hour_of_day
  - day_of_week
  - amount_to_balance_ratio
  - is_merchant_dest


In [5]:
df_final = drop_unused_columns(df_feat)

print(f"Columnas antes:  {df_feat.shape[1]}")
print(f"Columnas después: {df_final.shape[1]}")
print(f"\nColumnas finales:")
for c in df_final.columns:
    print(f"  - {c}")

Columnas antes:  15
Columnas después: 12

Columnas finales:
  - step
  - type
  - amount
  - oldbalanceOrg
  - newbalanceOrig
  - oldbalanceDest
  - newbalanceDest
  - isFraud
  - hour_of_day
  - day_of_week
  - amount_to_balance_ratio
  - is_merchant_dest


**Hallazgo**: el dataset queda con las columnas estrictamente necesarias para el modelado.

**Columnas preservadas**:

- `step`: unidad de tiempo.
- `type`: tipo de transacción (categórica).
- `amount`: monto.
- `oldbalanceOrg`, `newbalanceOrig`: saldos origen.
- `oldbalanceDest`, `newbalanceDest`: saldos destino.
- `isFraud`: etiqueta.
- `hour_of_day`, `day_of_week`: features temporales derivadas.
- `amount_to_balance_ratio`: feature derivada.
- `is_merchant_dest`: feature derivada.

**Decisiones éticas**:

- NO se crearon `error_balance_orig` ni `error_balance_dest` (data leakage).
- La señal de balance se preserva a través de las columnas originales de saldo.

---

## 4. Verificación de las features creadas

**Objetivo**: Validar que las features derivadas tienen distribuciones razonables y no introducen valores anómalos.

**Checks**:

- `hour_of_day` ∈ [0, 23]
- `day_of_week` ∈ [0, 6]
- `amount_to_balance_ratio` ≥ 0
- `is_merchant_dest` ∈ {0, 1}
- Sin valores nulos en ninguna feature.

In [6]:
print("Rango de features temporales:")
print(f"  hour_of_day:    min={df_final['hour_of_day'].min()}, max={df_final['hour_of_day'].max()}")
print(f"  day_of_week:    min={df_final['day_of_week'].min()}, max={df_final['day_of_week'].max()}")

print(f"\namount_to_balance_ratio:")
print(f"  min:    {df_final['amount_to_balance_ratio'].min():.4f}")
print(f"  max:    {df_final['amount_to_balance_ratio'].max():.4f}")
print(f"  mediana: {df_final['amount_to_balance_ratio'].median():.4f}")

print(f"\nis_merchant_dest:")
print(df_final['is_merchant_dest'].value_counts())

print(f"\nValores nulos por columna:")
nulls = df_final.isnull().sum()
if nulls.sum() == 0:
    print("  ✅ Sin valores nulos")
else:
    print(nulls[nulls > 0])

Rango de features temporales:
  hour_of_day:    min=0, max=23
  day_of_week:    min=0, max=6

amount_to_balance_ratio:
  min:    0.0000
  max:    92445516.6400
  mediana: 560.7207

is_merchant_dest:
is_merchant_dest
0    2770409
Name: count, dtype: int64

Valores nulos por columna:
  ✅ Sin valores nulos


---

## 5. Separación de features (X) y objetivo (y)

**Objetivo**: Separar el dataset en:

- `X`: matriz de features (todas las columnas excepto `isFraud`).
- `y`: vector de etiquetas (`isFraud`).

In [7]:
X, y = split_features_target(df_final, target=TARGET_COL)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nDistribución de y:")
print(y.value_counts(normalize=True).sort_index() * 100)

X shape: (2770409, 11)
y shape: (2770409,)

Distribución de y:
isFraud
0    99.703546
1     0.296454
Name: proportion, dtype: float64


---

## 6. Split estratificado 70/15/15

**Objetivo**: Dividir en train (70%), validación (15%) y test (15%).

**Estratificación**: se mantiene la proporción de fraude en cada split. Crítico con clases desbalanceadas.

**Por qué 3 conjuntos**:

- **Train**: para entrenar el modelo.
- **Validación**: para optimización de hiperparámetros (Optuna) y selección de modelo.
- **Test**: para evaluación final, solo al terminar (evitar overfitting al test).

In [8]:
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
)

print(f"Train:      {X_train.shape[0]:>12,} filas  ({y_train.mean()*100:.4f}% fraude)")
print(f"Val:        {X_val.shape[0]:>12,} filas  ({y_val.mean()*100:.4f}% fraude)")
print(f"Test:       {X_test.shape[0]:>12,} filas  ({y_test.mean()*100:.4f}% fraude)")
print(f"\nTotal:      {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]:>12,} filas")

Train:         1,939,286 filas  (0.2964% fraude)
Val:             415,561 filas  (0.2965% fraude)
Test:            415,562 filas  (0.2965% fraude)

Total:         2,770,409 filas


**Hallazgo**: el split estratificado preserva la proporción de fraude en los tres conjuntos.

**Verificación**: los tres porcentajes de fraude deben ser muy similares (dentro de ±0.01%).

**Decisión**: usar `X_train`/`y_train` para entrenamiento, `X_val`/`y_val` para Optuna, `X_test`/`y_test` para evaluación final.

---

## 7. Guardado de conjuntos procesados

**Objetivo**: Persistir los tres conjuntos en formato `.parquet` para reutilización en los Pasos 4, 5 y 6.

**Formato `.parquet`**:

- Más rápido de leer/escribir que CSV.
- Preserva tipos de datos.
- Ocupa menos espacio.
- Ideal para pipelines de ML.

In [12]:
# Guardar los splits como DataFrames combinados (X + y)
splits_to_save = [
    ("train", X_train, y_train),
    ("val",   X_val,   y_val),
    ("test",  X_test,  y_test),
]

for split_name, X_split, y_split in splits_to_save:
    df_split = X_split.copy()
    df_split[TARGET_COL] = y_split.values

    path = DATA_PROCESSED / f"{split_name}.parquet"
    df_split.to_parquet(path)
    print(f"✅ {path.name:<20} guardado ({len(df_split):,} filas)")

print(f"\nDirectorio: {DATA_PROCESSED}")

✅ train.parquet        guardado (1,939,286 filas)
✅ val.parquet          guardado (415,561 filas)
✅ test.parquet         guardado (415,562 filas)

Directorio: ..\data\processed
